<a href="https://colab.research.google.com/github/siva123h/NNDL/blob/main/Implement_Image_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import (
    MobileNetV2,
    preprocess_input,
    decode_predictions,
)
from tensorflow.keras.preprocessing import image

# Load pre-trained model (ImageNet)
# First time it will download weights from the internet
model = MobileNetV2(weights="imagenet")


def classify_image_by_path(img_path: str):
    """
    Takes an image path, predicts the object,
    and groups it into: animal / bird / flower / other
    """

    if not os.path.exists(img_path):
        print(f"Path not found: {img_path}")
        return None

    # 1. Load and preprocess image
    img = image.load_img(img_path, target_size=(224, 224))
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0)
    x = preprocess_input(x)

    # 2. Predict with the model
    preds = model.predict(x)
    decoded = decode_predictions(preds, top=5)[0]  # list of (id, name, score)

    print("Top predictions:")
    for (_, name, score) in decoded:
        print(f"- {name:20s}  prob={score:.3f}")

    # 3. Decide high-level category
    label_names = [name.lower() for (_, name, _) in decoded]

    # keywords for grouping
    bird_keywords = [
        "bird", "hen", "cock", "rooster", "ostrich", "flamingo",
        "partridge", "peacock", "sparrow", "parrot", "king_penguin", "goose"
    ]

    animal_keywords = [
        "dog", "cat", "puppy", "kitten", "horse", "cow", "bull", "sheep",
        "goat", "lion", "tiger", "elephant", "zebra", "bear", "monkey",
        "fox", "wolf", "leopard", "cheetah", "panda", "deer", "rabbit"
    ]

    flower_keywords = [
        "flower", "daisy", "sunflower", "tulip", "rose", "lotus",
        "orchid", "lily"
    ]

    def contains_any(label_list, keywords):
        return any(
            any(k in label for k in keywords)
            for label in label_list
        )

    if contains_any(label_names, bird_keywords):
        category = "bird"
    elif contains_any(label_names, animal_keywords):
        category = "animal"
    elif contains_any(label_names, flower_keywords):
        category = "flower"
    else:
        category = "other object"

    print(f"\nFinal category: {category}")
    return category, decoded


if __name__ == "__main__":
    # Ask user for image path
    path = input("Enter image path (e.g. C:/images/parrot.jpg): ")
    classify_image_by_path(path)

14536120/14536120 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Enter image path (e.g. C:/images/parrot.jpg): /content/download.jpeg
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
35363/35363 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Top predictions:
- Egyptian_cat          prob=0.850
- tabby                 prob=0.051
- tiger_cat             prob=0.013
- lynx                  prob=0.007
- Siamese_cat           prob=0.004

Final category: animal
